# 🔍 Stage 2 — Exploratory Data Analysis
### 🏦 Loan Default Predictor — Home Credit Dataset
---
**Understand the data through visualizations — missing values, distributions, correlations.**

> **Section 3**

```
Progress: ██░░░  Stage 2 of 5
```

← [Stage 1](stage_01_setup_and_data.ipynb)  |  [Stage 3](stage_03_preprocessing_and_features.ipynb) →

---
### 📋 What you will do in this stage:
- Analyze **missing values** — how much data is absent and what to do about it
- Explore the **age distribution** of defaulters vs non-defaulters
- Study **default rates by education** and **income type**
- Examine the power of **external credit scores** (EXT_SOURCE_1/2/3)
- Build a **correlation matrix** of key numerical features

⏱️ *Estimated time: 30–45 minutes*

---
> ⚠️ **Prerequisite:** This notebook depends on **Stage 1 (the dataset must be loaded as `df`)**.  
> Run the previous stage(s) first, **or** run the cell below to reload saved objects.


### ⚙️ Reload Cell
If you are starting fresh in this notebook (without running Stage 1 first),
run this cell to load the dataset. Otherwise skip it.


In [ ]:
# ── RELOAD: Run this only if you didn't just complete Stage 1 ──
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 12

df = pd.read_csv("data/raw/application_train.csv")
df["AGE_YEARS"] = (-df["DAYS_BIRTH"] / 365).round(1)

print(f"✅ Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"   Default rate: {df['TARGET'].mean():.2%}")

---
## Section 3 — Exploratory Data Analysis (EDA)

EDA means **getting to know your data** before modeling.  
We look for patterns, anomalies, and relationships between features and the target.

> 🔬 **Mini-example — Why EDA matters:**  
> If we discover that "clients with university education default much less", we know  
> education level will be a useful feature for the model.


### 3.1 — Missing Values

Real-world data is never perfect. Some columns have missing information.  
We need to understand *how much* is missing and *why* before deciding what to do.


In [ ]:
# Calculate percentage of missing values per column
missing = df.isnull().mean() * 100
missing = missing[missing > 0].sort_values(ascending=False)

print(f"Columns with missing data: {len(missing)} out of {df.shape[1]}")
print(f"
Top 15 columns with most missing data:")
print(missing.head(15).round(2).to_string())

In [ ]:
# Visualize missing data — top 30 columns
fig, ax = plt.subplots(figsize=(12, 6))

missing.head(30).plot(kind="bar", color="coral", edgecolor="black", ax=ax)
ax.set_title("Top 30 Columns by % Missing Values", fontsize=14, fontweight="bold")
ax.set_xlabel("Column Name")
ax.set_ylabel("% Missing")
ax.axhline(y=50, color="red", linestyle="--", label="50% threshold")
ax.legend()
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### 3.2 — Age Distribution

The column `DAYS_BIRTH` stores the client's age in **negative days** (days before application).  
We'll convert it to years for easier interpretation.

💡 **Why negative?** The dataset uses a convention where values are negative distances from the application date.


In [ ]:
# Convert DAYS_BIRTH to age in years (positive number)
df["AGE_YEARS"] = (-df["DAYS_BIRTH"] / 365).round(1)

print(f"Age range: {df['AGE_YEARS'].min()} to {df['AGE_YEARS'].max()} years")

In [ ]:
# Compare age distribution between defaulters and non-defaulters
fig, ax = plt.subplots(figsize=(11, 5))

df[df["TARGET"] == 0]["AGE_YEARS"].plot(kind="hist", bins=40, alpha=0.6,
                                         color="steelblue", label="Repaid", ax=ax)
df[df["TARGET"] == 1]["AGE_YEARS"].plot(kind="hist", bins=40, alpha=0.6,
                                         color="tomato", label="Defaulted", ax=ax)

ax.set_title("Age Distribution: Defaulters vs. Non-Defaulters", fontsize=14, fontweight="bold")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Show average age by target
print("Average age by loan outcome:")
print(df.groupby("TARGET")["AGE_YEARS"].mean().round(1).rename({0: "Repaid", 1: "Defaulted"}))

### 3.3 — Default Rate by Education Level

Do more educated clients default less? Let's find out.


In [ ]:
# Calculate default rate per education type
edu_default = (df.groupby("NAME_EDUCATION_TYPE")["TARGET"]
               .agg(["mean", "count"])
               .rename(columns={"mean": "default_rate", "count": "total"})
               .sort_values("default_rate", ascending=False))

edu_default["default_rate"] = (edu_default["default_rate"] * 100).round(2)
print(edu_default.to_string())

In [ ]:
# Bar chart — default rate by education
fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.bar(edu_default.index, edu_default["default_rate"],
              color=plt.cm.RdYlGn_r(edu_default["default_rate"] / edu_default["default_rate"].max()),
              edgecolor="black")

ax.set_title("Default Rate by Education Level", fontsize=14, fontweight="bold")
ax.set_xlabel("Education Type")
ax.set_ylabel("Default Rate (%)")
ax.set_xticklabels(edu_default.index, rotation=30, ha="right")

# Add value labels on bars
for bar, val in zip(bars, edu_default["default_rate"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f"{val:.1f}%", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()

### 3.4 — External Credit Scores

The dataset has 3 external risk scores: `EXT_SOURCE_1`, `EXT_SOURCE_2`, `EXT_SOURCE_3`.  
These come from external credit bureaus. Higher score = better creditworthiness.

These are typically some of the **most predictive features** in credit models.


In [ ]:
# Box plots — external scores by target
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, col in enumerate(["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]):
    df.boxplot(column=col, by="TARGET", ax=axes[i],
               boxprops=dict(color="navy"),
               medianprops=dict(color="red", linewidth=2))
    axes[i].set_title(f"{col}", fontsize=12, fontweight="bold")
    axes[i].set_xlabel("Target (0=Repaid, 1=Defaulted)")
    axes[i].set_ylabel("Score")

plt.suptitle("External Credit Scores vs. Loan Outcome", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

**Observation:** Clients who defaulted have **lower external scores** on average.  
This confirms these features will be valuable for our model.


### 3.5 — Default Rate by Income Type


In [ ]:
# Default rate by income type
income_default = (df.groupby("NAME_INCOME_TYPE")["TARGET"]
                  .agg(["mean", "count"])
                  .rename(columns={"mean": "default_rate", "count": "n"})
                  .sort_values("default_rate", ascending=True))

income_default["default_rate_pct"] = (income_default["default_rate"] * 100).round(2)
print(income_default[["default_rate_pct", "n"]].to_string())

In [ ]:
# Horizontal bar chart
fig, ax = plt.subplots(figsize=(10, 5))

ax.barh(income_default.index, income_default["default_rate_pct"],
        color="steelblue", edgecolor="black")
ax.set_title("Default Rate by Income Type", fontsize=14, fontweight="bold")
ax.set_xlabel("Default Rate (%)")

for i, val in enumerate(income_default["default_rate_pct"]):
    ax.text(val + 0.1, i, f"{val:.1f}%", va="center", fontsize=10)

plt.tight_layout()
plt.show()

### 3.6 — Correlation Matrix

Let's look at correlations between numerical features to:
1. Find features correlated with the target
2. Detect highly correlated features (multicollinearity)

💡 **Multicollinearity** means two features carry the same information.  
Including both doesn't help the model — it just adds noise.


In [ ]:
# Select key numerical columns for correlation analysis
num_cols = [
    "TARGET", "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY",
    "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3", "AGE_YEARS"
]

corr_matrix = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title("Correlation Matrix — Key Numerical Features", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# What correlates most with TARGET?
print("Feature correlations with TARGET (default):")
print(corr_matrix["TARGET"].drop("TARGET").sort_values().round(3).to_string())

---
## ✅ Stage 2 Complete!

Great work! Here is a summary of what you accomplished:

- Analyze **missing values** — how much data is absent and what to do about it
- Explore the **age distribution** of defaulters vs non-defaulters
- Study **default rates by education** and **income type**
- Examine the power of **external credit scores** (EXT_SOURCE_1/2/3)
- Build a **correlation matrix** of key numerical features

⏱️ *Estimated time: 30–45 minutes*

---
### ➡️ Next: 🛠️ Stage 3 — Preprocessing & Feature Engineering
**Clean the data, handle missing values, encode categories, scale features, and create new features.**

Open **`stage_03_preprocessing_and_features.ipynb`** to continue.
